<a href="https://colab.research.google.com/github/mcaaa9669/ai-math-assignment-1/blob/main/24011629_%EC%9D%B4%EC%84%9C%ED%98%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 센서 기반 AI 모델의 차원 축소 방식별 성능 비교 연구

**연구 목적:** 스마트폰 센서 데이터를 원본, 평균 풀링, PCA 방식으로 처리한 후, 행동 분류 모델의 예측 정확도, 연산 효율 및 노이즈 강건성을 비교한다.

**비교 조건**
- A: 원본 768차원
- B: 평균 풀링 192차원
- C: PCA 192차원

**실험 흐름**

데이터 준비 → 차원 축소 및 검산 → 분류 모델 학습 → 성능 진단 → 연산 효율 및 노이즈 강건성 분석

**핵심 원칙:** 입력 처리 방식 외의 학습, 평가 조건을 최대한 동일하게 유지하여 차원 축소 방식의 영향을 분석한다.

## 01. 데이터 준비하기: UCI HAR 센서 신호와 정답



가속도 3축과 자이로스코프 3축의 신호를 불러온다. 각 샘플은 128시점 × 6채널이며, 분류 정답은 걷기, 계단 오르기, 계단 내려가기, 앉아있기, 서있기, 누워있기 로 총 6개이다.

In [ ]:

import io
import zipfile
import urllib.request
from pathlib import Path

import numpy as np

# 1. UCI HAR 데이터셋 다운로드
url = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"

data_dir = Path("/content/har_data")
data_dir.mkdir(parents=True, exist_ok=True)

archive_path = data_dir / "har.zip"

if not archive_path.exists():
    urllib.request.urlretrieve(url, archive_path)

# 2. 압축 해제 (내부 ZIP 파일이 있는 경우도 처리)
def extract_zip(file_path, target_dir):
    with zipfile.ZipFile(file_path) as archive:
        archive.extractall(target_dir)
        for name in archive.namelist():
            if name.lower().endswith(".zip"):
                nested = target_dir / name
                if nested.exists():
                    with zipfile.ZipFile(nested) as inner:
                        inner.extractall(target_dir)

extract_zip(archive_path, data_dir)

# 3. 센서 파일이 들어있는 폴더 찾기
candidates = list(data_dir.rglob("total_acc_x_train.txt"))
assert candidates, "센서 신호 파일을 찾지 못했습니다."

signal_dir = candidates[0].parent.parent
dataset_root = signal_dir.parent

channels = [
    "total_acc_x",
    "total_acc_y",
    "total_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z"
]

def load_signals(split):
    folder = dataset_root / split / "Inertial Signals"
    arrays = [
        np.loadtxt(folder / f"{name}_{split}.txt",
                   dtype=np.float32)
        for name in channels
    ]
    return np.stack(arrays, axis=-1)

# 4. 학습 및 평가 데이터
X_train = load_signals("train")
X_test = load_signals("test")

y_train = np.loadtxt(
    dataset_root / "train" / "y_train.txt",
    dtype=int
) - 1

y_test = np.loadtxt(
    dataset_root / "test" / "y_test.txt",
    dtype=int
) - 1

# 5. 입력 배열 구조 확인
print("학습 센서 데이터:", X_train.shape)
print("평가 센서 데이터:", X_test.shape)
print("학습 정답:", y_train.shape)
print("평가 정답:", y_test.shape)
print("분류 클래스:", np.unique(y_train))
print("센서 채널:", channels)

assert X_train.shape == (7352, 128, 6)
assert X_test.shape == (2947, 128, 6)


학습 센서 데이터: (7352, 128, 6)
평가 센서 데이터: (2947, 128, 6)
학습 정답: (7352,)
평가 정답: (2947,)
분류 클래스: [0 1 2 3 4 5]
센서 채널: ['total_acc_x', 'total_acc_y', 'total_acc_z', 'body_gyro_x', 'body_gyro_y', 'body_gyro_z']


## 02. 입력 차원 확인: 원본과 평균 풀링크기 확인

원본은 128 x 6을 펼치고, 평균 풀링은 채널별로 연속 4시점을 평균내 192차원을 만든다.

In [ ]:
# 02. 원본과 4시점 평균 풀링의 입력 크기 확인

def average_pooling(values):
    return values.reshape(-1, 32, 4, 6).mean(axis=2).reshape(-1, 192)


X_original = X_train.reshape(len(X_train), -1)
X_pooled = average_pooling(X_train)
print('원본 입력:', X_original.shape)
print('평균 풀링 입력:', X_pooled.shape)
print('PCA 목표 차원: 192')

원본 입력: (7352, 768)
평균 풀링 입력: (7352, 192)
PCA 목표 차원: 192


## 03. PCA 구현

학습 데이터로 표준화 기준과 공분산을 구한다. 분산이 큰 순서대로 선택한 주성분 192개에 학습, 평가 입력을 투영한다.

In [ ]:
# 03. 두 데이터 분할에서 공통으로 사용하는 PCA와 표준화 함수

def standardize_from_train(train, evaluation):
    mean = train.mean(axis=0)
    scale = train.std(axis=0)
    scale[scale < 1e-8] = 1.0
    return (train - mean) / scale, (evaluation - mean) / scale, (mean, scale)


def fit_sensor_pca(train, evaluation, n_components=192):
    train_flat = train.reshape(len(train), -1).astype(np.float64)
    eval_flat = evaluation.reshape(len(evaluation), -1).astype(np.float64)
    train_std, eval_std, sensor_stats = standardize_from_train(train_flat, eval_flat)
    pca_mean = train_std.mean(axis=0)
    centered = train_std - pca_mean
    # 교안에서 사용한 1/N 공분산 정의
    covariance = centered.T @ centered / len(centered)
    eigenvalues, eigenvectors = np.linalg.eigh(covariance)
    order = np.argsort(eigenvalues)[::-1]
    selected = order[:n_components]
    basis = eigenvectors[:, selected]
    return {
        'standardized': (train_std, eval_std),
        'projected': (centered @ basis, (eval_std - pca_mean) @ basis),
        'sensor_stats': sensor_stats, 'pca_mean': pca_mean, 'basis': basis,
        'covariance': covariance, 'eigenvalues': eigenvalues, 'order': order,
        'explained_ratio': eigenvalues[selected].sum() / eigenvalues.sum(),
    }


# 공식 학습: 테스트 분할의 초기 실험
initial_pca = fit_sensor_pca(X_train, X_test)
train_std, test_std = initial_pca['standardized']
X_pca_train, X_pca_test = initial_pca['projected']
cov, Vk = initial_pca['covariance'], initial_pca['basis']
eigenvalues, indices = initial_pca['eigenvalues'], initial_pca['order']
print('공분산 행렬:', cov.shape)
print('PCA 학습 입력:', X_pca_train.shape)
print('PCA 평가 입력:', X_pca_test.shape)
print(f"분산 보존 비율: {initial_pca['explained_ratio']:.2%}")

공분산 행렬: (768, 768)
PCA 학습 입력: (7352, 192)
PCA 평가 입력: (2947, 192)
분산 보존 비율: 98.31%


## 04. PCA 검산: 직교성,고유값 방정식,손계산 예제

계산한 주성분이 수학적 조건을 만족하는지 확인한다. 또한 작은 행렬의 손계산 결과도 코드와 비교한다.

In [ ]:
# 04. PCA 기저의 직교성과 고유값 방정식 검산

def pca_residuals(pca_fit):
    basis = pca_fit['basis']
    selected = pca_fit['order'][:basis.shape[1]]
    values = pca_fit['eigenvalues'][selected]
    identity = np.eye(basis.shape[1])
    orth_error = np.max(np.abs(basis.T @ basis - identity))
    eigen_error = np.max(np.abs(pca_fit['covariance'] @ basis - basis * values))
    return orth_error, eigen_error


error, eigen_error = pca_residuals(initial_pca)
assert error < 1e-10 and eigen_error < 1e-10
assert X_pca_train.shape == (len(X_train), 192)
assert X_pca_test.shape == (len(X_test), 192)
print(f'고유벡터 직교성 오차: {error:.2e}')
print(f'고유값 방정식 오차: {eigen_error:.2e}')
print('PCA 차원 검증 완료')

# 손계산 예제: 공분산 [[1, 1], [1, 1]]의 최대 고유값은 2.
example = np.array([[-1.0, -1.0], [1.0, 1.0]])
example_fit = fit_sensor_pca(example, example, n_components=1)
np.testing.assert_allclose(example_fit['covariance'], np.ones((2, 2)))
np.testing.assert_allclose(example_fit['eigenvalues'][-1], 2.0)
np.testing.assert_allclose(np.abs(example_fit['projected'][0]).ravel(), np.sqrt(2))
print('손계산 PCA 검산: 고유값 2, 투영값의 절댓값 √2 확인')

고유벡터 직교성 오차: 3.11e-15
고유값 방정식 오차: 1.03e-13
PCA 차원 검증 완료
손계산 PCA 검산: 고유값 2, 투영값의 절댓값 √2 확인


## 05. 초기 검산 출력 기록

04번의 수치 오차를 더 많은 자릿수로 남긴 기록이다. 현재 검산 결과는 04번과 10번에서 확인할 수 있다.

In [ ]:

# 실제 오차를 지수 표기법으로 확인

print(f"고유벡터 직교성 오차: {error:.16e}")
print(f"고유값 방정식 오차: {eigen_error:.16e}")

print("실제로 오차가 정확히 0인가?")
print("직교성 오차:", error == 0)
print("고유값 방정식 오차:", eigen_error == 0)


고유벡터 직교성 오차: 3.1086244689504383e-15
고유값 방정식 오차: 1.0302869668521453e-13
실제로 오차가 정확히 0인가?
직교성 오차: False
고유값 방정식 오차: False


## 06. 비교 입력 구성 — A 원본·B 평균 풀링·C PCA

세 입력 방식은 같은 표준화 신호에서 시작한다. 각 방식의 변환 뒤에도 학습 데이터의 통계로 표준화해 분류기 입력을 준비한다.

In [ ]:
# 06. 세 입력 방식에 동일한 처리 순서 적용

def build_input_datasets(pca_fit):
    train_std, eval_std = pca_fit['standardized']
    pca_train, pca_eval = pca_fit['projected']
    inputs = [
        ('A_original', train_std, eval_std, 768),
        ('B_pooling', average_pooling(train_std), average_pooling(eval_std), 192),
        ('C_PCA', pca_train, pca_eval, 192),
    ]
    datasets, stats = {}, {}
    for name, train, evaluation, dimension in inputs:
        train, evaluation, stats[name] = standardize_from_train(train, evaluation)
        assert train.shape == (len(train_std), dimension)
        assert evaluation.shape == (len(eval_std), dimension)
        assert np.isfinite(train).all() and np.isfinite(evaluation).all()
        datasets[name] = (train.astype(np.float32), evaluation.astype(np.float32))
    return datasets, stats


datasets, branch_stats = build_input_datasets(initial_pca)
for name, (train, evaluation) in datasets.items():
    print(f'{name}: 학습 {train.shape}, 평가 {evaluation.shape}')

A_original: 학습 (7352, 768), 평가 (2947, 768)
B_pooling: 학습 (7352, 192), 평가 (2947, 192)
C_PCA: 학습 (7352, 192), 평가 (2947, 192)


## 07. 초기 분류 실험 — Softmax 구현과 공식 테스트

교차 엔트로피와 경사하강법으로 Softmax 회귀를 구현한다. 공식 테스트 세트에서 동일한 학습 조건으로 A, B, C의 정확도와 Macro F1을 비교한다.

In [ ]:
# 07. Softmax 회귀 공통 구현

import numpy as np
from sklearn.metrics import accuracy_score, f1_score

# 1. Softmax 함수
def softmax(scores):
    scores = scores - np.max(scores, axis=1, keepdims=True)
    exp_scores = np.exp(scores)
    return exp_scores / np.sum(exp_scores, axis=1, keepdims=True)

# 2. 교차 엔트로피 손실 및 기울기
def loss_and_gradients(X, y, W, b, reg=1e-4):
    n = len(y)

    scores = X @ W + b
    probs = softmax(scores)

    # 교차 엔트로피
    loss = -np.mean(np.log(probs[np.arange(n), y] + 1e-12))

    # L2 정규화
    loss += 0.5 * reg * np.sum(W ** 2)

    # 손실 함수의 기울기
    grad = probs.copy()
    grad[np.arange(n), y] -= 1
    grad /= n

    dW = X.T @ grad + reg * W
    db = np.sum(grad, axis=0)

    return loss, dW, db

# 3. 경사하강법으로 모델 학습
def train_softmax(X, y, epochs=120, lr=0.05, batch_size=256):
    rng = np.random.default_rng(42)

    n, d = X.shape
    num_classes = 6

    W = np.zeros((d, num_classes), dtype=np.float32)
    b = np.zeros(num_classes, dtype=np.float32)

    for epoch in range(epochs):
        indices = rng.permutation(n)

        for start in range(0, n, batch_size):
            batch = indices[start:start + batch_size]

            _, dW, db = loss_and_gradients(
                X[batch], y[batch], W, b
            )

            W -= lr * dW
            b -= lr * db

        if (epoch + 1) % 40 == 0:
            loss, _, _ = loss_and_gradients(X, y, W, b)
            print(f"  Epoch {epoch+1}: Loss = {loss:.4f}")

    return W, b

In [ ]:
# 공식 학습·테스트 분할의 초기 Softmax 결과
# 4. 모델별 학습 및 평가
results = {}
models = {}

for name, (Xtr, Xte) in datasets.items():
    print(f"\n===== {name} =====")

    W, b = train_softmax(Xtr, y_train)
    models[name] = (W, b)

    probs = softmax(Xte @ W + b)
    predictions = np.argmax(probs, axis=1)

    acc = accuracy_score(y_test, predictions)
    f1 = f1_score(y_test, predictions, average="macro")

    results[name] = {"Accuracy": acc, "Macro F1": f1}

    print(f"정확도: {acc:.4f}")
    print(f"Macro F1: {f1:.4f}")

print("\n===== 최종 비교 결과 =====")
for name, values in results.items():
    print(
        f"{name}: "
        f"Accuracy={values['Accuracy']:.4f}, "
        f"Macro F1={values['Macro F1']:.4f}"
    )


===== A_original =====
  Epoch 40: Loss = 0.8458
  Epoch 80: Loss = 0.8281
  Epoch 120: Loss = 0.8140
정확도: 0.5741
Macro F1: 0.5376

===== B_pooling =====
  Epoch 40: Loss = 0.8946
  Epoch 80: Loss = 0.8858
  Epoch 120: Loss = 0.8801
정확도: 0.5545
Macro F1: 0.5255

===== C_PCA =====
  Epoch 40: Loss = 0.9445
  Epoch 80: Loss = 0.9179
  Epoch 120: Loss = 0.9082
정확도: 0.5633
Macro F1: 0.5373

===== 최종 비교 결과 =====
A_original: Accuracy=0.5741, Macro F1=0.5376
B_pooling: Accuracy=0.5545, Macro F1=0.5255
C_PCA: Accuracy=0.5633, Macro F1=0.5373


## 08. 초기 결과 진단 — 학습·테스트 격차와 오분류

학습과 공식 테스트의 정확도 차이를 보고, 원본 입력의 혼동행렬로 어떤 행동을 헷갈리는지 확인한다.

In [ ]:

# 08. 모델 성능 진단
# 학습 정확도와 평가 정확도를 비교하여 일반화 문제 확인

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("===== 모델별 학습/평가 성능 비교 =====")

for name, (Xtr, Xte) in datasets.items():
    W, b = models[name]

    train_pred = np.argmax(Xtr @ W + b, axis=1)
    test_pred = np.argmax(Xte @ W + b, axis=1)

    train_acc = accuracy_score(y_train, train_pred)
    test_acc = accuracy_score(y_test, test_pred)

    gap = train_acc - test_acc

    print(f"\n{name}")
    print(f"학습 정확도: {train_acc:.2%}")
    print(f"평가 정확도: {test_acc:.2%}")
    print(f"정확도 격차: {gap * 100:.2f}%p")


# 원본 모델의 행동별 성능 확인
Xtr, Xte = datasets["A_original"]
W, b = models["A_original"]

pred = np.argmax(Xte @ W + b, axis=1)

activity_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

print("\n===== 원본 모델 행동별 성능 =====")
print(classification_report(
    y_test,
    pred,
    target_names=activity_names,
    zero_division=0
))

print("\n===== 오분류 행렬 =====")
print(confusion_matrix(y_test, pred))


===== 모델별 학습/평가 성능 비교 =====

A_original
학습 정확도: 70.29%
평가 정확도: 57.41%
정확도 격차: 12.88%p

B_pooling
학습 정확도: 66.49%
평가 정확도: 55.45%
정확도 격차: 11.04%p

C_PCA
학습 정확도: 66.72%
평가 정확도: 56.33%
정확도 격차: 10.39%p

===== 원본 모델 행동별 성능 =====
                    precision    recall  f1-score   support

           WALKING       0.37      0.18      0.24       496
  WALKING_UPSTAIRS       0.46      0.53      0.49       471
WALKING_DOWNSTAIRS       0.47      0.22      0.30       420
           SITTING       0.60      0.76      0.67       491
          STANDING       0.44      0.66      0.53       532
            LAYING       1.00      1.00      1.00       537

          accuracy                           0.57      2947
         macro avg       0.56      0.56      0.54      2947
      weighted avg       0.56      0.57      0.55      2947


===== 오분류 행렬 =====
[[ 88 128  68  65 147   0]
 [ 59 249  38  40  85   0]
 [ 83  59  93  59 126   0]
 [  0  24   0 374  93   0]
 [  5  87   0  89 351   0]
 [  0   0   0   0   

## 09. 내부 검증 설계 — 참가자 단위 분할

공식 학습 세트에서 16명은 학습, 5명은 내부 검증에 사용한다. 참가자가 겹치지 않도록 나눈다.

In [ ]:

# 09. 참가자 단위 검증 데이터 분리
# 같은 참가자의 데이터가 학습과 검증에 동시에 들어가지 않도록 설정

from sklearn.model_selection import GroupShuffleSplit

subject_train = np.loadtxt(
    dataset_root / "train" / "subject_train.txt",
    dtype=int
)

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(X_train, y_train, groups=subject_train)
)

train_subjects = set(subject_train[train_idx])
val_subjects = set(subject_train[val_idx])

print("학습 샘플 수:", len(train_idx))
print("검증 샘플 수:", len(val_idx))
print("학습 참가자 수:", len(train_subjects))
print("검증 참가자 수:", len(val_subjects))
print("참가자 중복 여부:", bool(train_subjects & val_subjects))

assert train_subjects.isdisjoint(val_subjects)

print("학습/검증 참가자 분리 완료")


학습 샘플 수: 5551
검증 샘플 수: 1801
학습 참가자 수: 16
검증 참가자 수: 5
참가자 중복 여부: False
학습/검증 참가자 분리 완료


## 10. 학습 데이터 기준 전처리 및 PCA 재계산

09번의 참가자 분할을 유지한다. 원본 센서의 표준화 통계, PCA 주성분, 입력 방식별 표준화 통계는 학습 참가자 데이터로만 구한다. 검증 데이터에는 학습에서 구한 값을 그대로 적용한다. A는 원본 768차원, B는 4시점 평균 풀링 192차원, C는 PCA 192차원이다. 공식 테스트 세트는 이번 실험에 사용하지 않는다.

In [ ]:
# 10. 내부 참가자 분할에서 전처리 재적합 및 PCA 검산
assert set(subject_train[train_idx]).isdisjoint(subject_train[val_idx])
y_fit_10, y_val_10 = y_train[train_idx], y_train[val_idx]
assert np.array_equal(np.unique(y_fit_10), np.arange(6))
assert np.array_equal(np.unique(y_val_10), np.arange(6))

pca_fit_10 = fit_sensor_pca(X_train[train_idx], X_train[val_idx])
datasets_10, branch_stats_10 = build_input_datasets(pca_fit_10)
fit_std_10, val_std_10 = pca_fit_10['standardized']
sensor_stats_10 = pca_fit_10['sensor_stats']
pca_mean_10, pca_basis_10 = pca_fit_10['pca_mean'], pca_fit_10['basis']

orth_error_10, eigen_error_10 = pca_residuals(pca_fit_10)
assert orth_error_10 < 1e-10 and eigen_error_10 < 1e-10
for name, (train, validation) in datasets_10.items():
    print(f'{name}: 학습 {train.shape}, 검증 {validation.shape}')
print(f"PCA 192개 주성분의 누적 설명 분산 비율: {pca_fit_10['explained_ratio']:.2%}")
print(f'직교성 최대 오차: {orth_error_10:.2e}')
print(f'고유값 방정식 최대 오차: {eigen_error_10:.2e}')

A_original: 학습 (5551, 768), 검증 (1801, 768)
B_pooling: 학습 (5551, 192), 검증 (1801, 192)
C_PCA: 학습 (5551, 192), 검증 (1801, 192)
PCA 192개 주성분의 누적 설명 분산 비율: 98.35%
직교성 최대 오차: 2.22e-15
고유값 방정식 최대 오차: 1.88e-13


## 11. 정상 입력 성능 비교와 MLP 보조 실험

**주 실험.** 09번의 같은 검증 참가자에서 A·B·C의 Softmax 회귀 결과를 비교한다. Accuracy는 전체 정답 비율이고, Macro F1은 여섯 행동의 F1을 같은 비중으로 평균한 값이다. 07~08번은 UCI HAR 공식 테스트를 사용한 초기 실험이다.

**보조 실험.** 같은 입력 방식에서 기존 Softmax 회귀와 은닉층 하나(64개 뉴런, ReLU)를 둔 MLP를 비교한다. 표의 개선폭은 모델 구조와 최적화 방식이 함께 달라진 결과다. 행동별 F1으로 걷기와 계단 이동에서 어떤 변화가 있었는지 확인한다.

**다음 실험.** 정상 입력에서 학습한 Softmax를 그대로 사용해 A·B·C의 전처리 포함 추론 비용과 노이즈·센서 결측에 따른 성능 변화를 비교한다.

In [ ]:
# 11. 동일한 참가자 분할로 기존 모델과 비선형 모델 비교
from IPython.display import display
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from sklearn.neural_network import MLPClassifier
from threadpoolctl import threadpool_limits

labels_11 = np.arange(6)
activity_names_11 = ['WALKING', 'WALKING_UPSTAIRS', 'WALKING_DOWNSTAIRS',
                     'SITTING', 'STANDING', 'LAYING']
results_11, models_11, predictions_11 = [], {}, {}

with threadpool_limits(limits=2, user_api='blas'):
    for name, (fit, val) in datasets_10.items():
        print(f'학습 중: {name}', flush=True)
        # 07번의 직접 구현과 기본 학습 설정을 유지한다.
        W, b = train_softmax(fit, y_fit_10)
        softmax_pred = np.argmax(val @ W + b, axis=1)

        # 검증 데이터와 별개로 학습 데이터에서 120 epoch를 수행한다.
        mlp = MLPClassifier(hidden_layer_sizes=(64,), activation='relu',
                            solver='adam', alpha=1e-4, batch_size=256,
                            learning_rate_init=0.001, random_state=42)
        for _ in range(120):
            mlp.partial_fit(fit, y_fit_10, classes=labels_11)
        mlp_pred = mlp.predict(val)
        models_11[name] = {'Softmax': (W, b), 'MLP': mlp}
        predictions_11[name] = {'Softmax': softmax_pred, 'MLP': mlp_pred}

        for model_name, pred in predictions_11[name].items():
            results_11.append({
                '입력': name, '모델': model_name,
                'Accuracy': accuracy_score(y_val_10, pred),
                'Macro F1': f1_score(y_val_10, pred, labels=labels_11,
                                     average='macro', zero_division=0),
            })

comparison_11 = pd.DataFrame(results_11).pivot(
    index='입력', columns='모델', values=['Accuracy', 'Macro F1']
)
for metric in ['Accuracy', 'Macro F1']:
    comparison_11[metric, 'MLP 개선폭 (%p)'] = 100 * (
        comparison_11[metric, 'MLP'] - comparison_11[metric, 'Softmax']
    )
print('검증 성능 비교 (Accuracy와 Macro F1은 0~1)')
display(comparison_11.sort_index(axis=1).round(4))

per_activity_11 = pd.DataFrame({
    (name, model): f1_score(y_val_10, pred, labels=labels_11,
                            average=None, zero_division=0)
    for name, predictions in predictions_11.items()
    for model, pred in predictions.items()
}, index=activity_names_11)
print('행동별 검증 F1')
display(per_activity_11.round(4))

학습 중: A_original
  Epoch 40: Loss = 0.8580
  Epoch 80: Loss = 0.8361
  Epoch 120: Loss = 0.8155
학습 중: B_pooling
  Epoch 40: Loss = 0.9145
  Epoch 80: Loss = 0.9042
  Epoch 120: Loss = 0.8977
학습 중: C_PCA
  Epoch 40: Loss = 0.9619
  Epoch 80: Loss = 0.9326
  Epoch 120: Loss = 0.9222
검증 성능 비교 (Accuracy와 Macro F1은 0~1)


Accuracy                      Macro F1                     
모델              MLP MLP 개선폭 (%p) Softmax      MLP MLP 개선폭 (%p) Softmax
입력                                                                    
A_original   0.8939      24.2643  0.6513   0.8915      27.4220  0.6173
B_pooling    0.9167      26.9295  0.6474   0.9137      30.1914  0.6118
C_PCA        0.8479      19.8223  0.6496   0.8401      22.8328  0.6118

행동별 검증 F1


A_original         B_pooling           C_PCA        
                      Softmax     MLP   Softmax     MLP Softmax     MLP
WALKING                0.3475  0.8926    0.3527  0.9126  0.3194  0.7931
WALKING_UPSTAIRS       0.4802  0.8682    0.5317  0.9084  0.4560  0.7404
WALKING_DOWNSTAIRS     0.3369  0.8571    0.3118  0.8664  0.3387  0.7412
SITTING                0.9114  0.8603    0.8960  0.9219  0.9156  0.9036
STANDING               0.6278  0.8924    0.5785  0.8820  0.6408  0.8713
LAYING                 1.0000  0.9784    1.0000  0.9910  1.0000  0.9909